# exp03 - Rossmann: ablasi kebocoran fitur `Customers`

## Masalah

Pada `our_study_rosman.ipynb`, matriks fitur dibentuk dengan
`X_train = train.drop(columns=["Sales_log", "Date"])`, sehingga kolom **`Customers`
ikut menjadi prediktor `Sales` pada hari yang sama**.

Ini bukan forecasting yang sah. Pada kompetisi Rossmann Store Sales, `Customers`
tidak tersedia pada `test.csv`: untuk meramalkan penjualan besok, jumlah pengunjung
besok belum diketahui. Model yang memakai `Customers` kontemporer sesungguhnya
menjawab pertanyaan yang berbeda ("berapa penjualan bila saya sudah tahu berapa
orang yang datang"), dan angkanya tidak dapat dibandingkan dengan metode lain yang
tidak memakai informasi tersebut.

Sel berikutnya memverifikasi klaim ini langsung dari berkas kompetisi, bukan dari
ingatan - bukti yang dapat diperiksa reviewer.

## Rancangan ablasi

| Varian | Perlakuan `Customers` | Sah sebagai forecasting? |
|---|---|---|
| `V0_customers_contemporaneous` | dipakai apa adanya (pipeline lama) | **Tidak** - hanya dilaporkan sebagai batas atas/acuan kebocoran |
| `V1_customers_dropped` | dibuang seluruhnya | Ya |
| `V2_customers_lagged` | diganti riwayat per toko: `Customers_lag1`, `Customers_roll7` | Ya |
| `V3_sales_lagged` | `V1` + riwayat penjualan per toko: `Sales_lag1`, `Sales_roll7` | Ya |

Keempat varian memakai **himpunan baris, split tanggal, protokol tuning, dan seed
yang identik**; satu-satunya yang berbeda adalah kolom fitur. Baris yang hilang akibat
pembentukan lag dibuang satu kali di awal untuk semua varian, sehingga selisih metrik
murni berasal dari perlakuan fitur.

In [ ]:
import sys, os, json, warnings
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.experiments import protocol as P

P.set_global_seed()
EXPERIMENT = "exp03_rossmann_leakage_ablation"
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
print("Lingkungan:", P.environment_stamp())

## 1. Bukti: `Customers` tidak ada pada berkas test kompetisi

In [ ]:
train_cols = pd.read_csv("../data/raw/rossmann/train.csv", nrows=0).columns.tolist()
test_cols  = pd.read_csv("../data/raw/rossmann/test.csv",  nrows=0).columns.tolist()

print("Kolom train.csv :", train_cols)
print("Kolom test.csv  :", test_cols)
print()
print("Tersedia di train tetapi TIDAK di test:",
      sorted(set(train_cols) - set(test_cols)))
assert "Customers" not in test_cols, "asumsi berubah - periksa kembali berkas data"
print("\nTerverifikasi: 'Customers' hanya ada pada data historis, bukan pada horizon"
      " yang diramalkan.")

## 2. Membangun kerangka fitur

Perbedaan yang disengaja terhadap notebook lama:

* lag dan rolling per toko dihitung **sebelum** penyaringan `Open == 1`, sehingga
  kalendernya benar (hari tutup tetap menggeser riwayat);
* semua fitur riwayat sudah di-`shift(1)` - tidak ada nilai hari-H yang masuk;
* split memakai **tanggal**, bukan indeks baris. Split berbasis indeks pada data panel
  1.115 toko memotong satu tanggal menjadi dua blok, sehingga sebagian baris pada
  tanggal batas berada di train dan sisanya di test.

In [ ]:
frame = P.build_rossmann_frame("../data/raw/rossmann/train.csv",
                               "../data/raw/rossmann/store.csv")
print("Bentuk kerangka setelah rekayasa fitur & penyaringan:", frame.shape)
print("Rentang tanggal:", frame["Date"].min().date(), "-", frame["Date"].max().date())
print("Jumlah toko:", frame["Store"].nunique())

datasets = {v: P.build_rossmann_dataset(frame, v, target="log1p")
            for v in P.CUSTOMER_VARIANTS}

split_table = pd.DataFrame([{**d.describe(),
                             "features": ", ".join(d.feature_names)}
                            for d in datasets.values()])
split_table.to_csv(f"../results/{EXPERIMENT}_splits.csv", index=False)
split_table[["feature_set", "n_features", "n_train", "n_val", "n_test",
             "train_start", "train_end", "val_end", "test_start", "test_end"]]

## 3. Menjalankan model pada setiap varian

Target adalah `log1p(Sales)`; metrik dilaporkan pada **kedua skala** secara terpisah
(`test_*` = skala log, `orig_*` = skala asli setelah `expm1`). Kedua blok tidak pernah
dicampur dalam satu tabel tanpa penanda - ini persis kritik reviewer nomor 2.

Baseline `SeasonalNaive` adalah median penjualan historis per (Toko x Hari x Promo)
dari blok training aktif. Baseline ini wajib: tanpanya tidak ada cara menilai apakah
keunggulan model berasal dari pemodelan atau sekadar dari struktur toko-hari.

**Anggaran komputasi.** `GRID_XGB_ROSSMANN` berisi 12 konfigurasi. Setiap model
menjalankan 12 fit pada blok train (~590 ribu baris) untuk tuning, ditambah 1 fit pada
train+val. Perkiraan beberapa jam pada CPU multi-core. Untuk uji cepat, aktifkan
`QUICK_RUN` di bawah.

In [ ]:
QUICK_RUN = False   # True = subset 100 toko + grid minimal, untuk memvalidasi pipeline

if QUICK_RUN:
    stores = np.sort(frame["Store"].unique())[:100]
    frame_run = frame[frame["Store"].isin(stores)].reset_index(drop=True)
    datasets = {v: P.build_rossmann_dataset(frame_run, v, target="log1p")
                for v in P.CUSTOMER_VARIANTS}
    XGB_GRID = {"n_estimators": [300], "max_depth": [6], "learning_rate": [0.1],
                "subsample": [0.8], "colsample_bytree": [0.8], "max_bin": [256]}
else:
    XGB_GRID = P.GRID_XGB_ROSSMANN

MODELS = [
    ("LR",                P.fp_linear_regression, None),
    ("XGBoost",           P.fp_xgboost,           XGB_GRID),
    ("LR+XGB (average)",  P.fp_lr_xgb_average,    XGB_GRID),
    ("LR-XGB (residual)", P.fp_lr_xgb_residual,   XGB_GRID),
]

rows = []
for variant, d in datasets.items():
    rows.append(P.rossmann_seasonal_naive(d, inverse_transform=np.expm1))
    for name, fn, grid in MODELS:
        rows.append(P.run_model(name, fn, d, grid, inverse_transform=np.expm1))
        print(f"  {variant:32s} {name:20s} selesai", flush=True)

results = P.save_results(rows, EXPERIMENT)
print(f"\n{len(results)} baris ditulis ke ../results/{EXPERIMENT}.csv")

## 4. Tabel utama - dampak kebocoran

In [ ]:
cols_log  = ["feature_set", "model", "n_features", "test_RMSE", "test_MAE", "test_R2"]
cols_orig = ["feature_set", "model", "orig_RMSE", "orig_MSE", "orig_MAE",
             "orig_RMSPE", "orig_R2"]

print("=== SKALA LOG (target log1p, tanpa inverse) ===")
display(results[cols_log].round(5).to_string(index=False))
print("\n=== SKALA ASLI (setelah expm1) ===")
display(results[cols_orig].round(4).to_string(index=False))

In [ ]:
# Berapa besar kebocoran menaikkan skor? Selisih terhadap varian bebas-kebocoran.
pivot = results.pivot_table(index="model", columns="feature_set", values="orig_RMSE")
pivot = pivot[[c for c in P.CUSTOMER_VARIANTS if c in pivot.columns]]
pivot["inflasi V1 vs V0 (%)"] = (
    (pivot["V1_customers_dropped"] - pivot["V0_customers_contemporaneous"])
    / pivot["V1_customers_dropped"] * 100)
print("RMSE skala asli per model x varian, dan berapa persen RMSE 'membaik' "
      "hanya karena kebocoran:")
pivot.round(3)

## 5. Bukti mekanistis: importance `Customers` pada varian bocor

Bila `Customers` mendominasi importance pada `V0`, itu menegaskan bahwa performa yang
dilaporkan sebelumnya digerakkan oleh informasi yang tidak tersedia saat peramalan.

In [ ]:
d0 = datasets["V0_customers_contemporaneous"]
best_v0 = results[(results.feature_set == "V0_customers_contemporaneous") &
                  (results.model == "LR-XGB (residual)")].iloc[0]
params = json.loads(best_v0["params"]) if isinstance(best_v0["params"], str) else {}

from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(d0.X_trainval, d0.y_trainval)
resid = d0.y_trainval - lr.predict(d0.X_trainval)
xgb_model = P.make_xgb(params).fit(d0.X_trainval, resid)

gains = getattr(xgb_model, "feature_importances_", None)
if gains is None:
    booster = xgb_model.get_booster()
    score = booster.get_score(importance_type="gain")
    gains = np.array([score.get(f"f{i}", 0.0) for i in range(len(d0.feature_names))])
importance = (pd.DataFrame({"feature": d0.feature_names, "gain": gains})
              .sort_values("gain", ascending=False).reset_index(drop=True))
display(importance.head(10).round(4))

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
top = importance.head(10).iloc[::-1]
ax.barh(top["feature"], top["gain"],
        color=["#c0392b" if f == "Customers" else "#7f8c8d" for f in top["feature"]])
ax.set_xlabel("Feature importance (gain)")
ax.set_title("V0 (bocor): kontribusi 'Customers' pada model residual")
plt.tight_layout(); plt.show()

## 6. Uji Diebold-Mariano terhadap baseline naif

Pertanyaan yang harus dijawab naskah: setelah kebocoran dihilangkan, apakah metode
usulan masih mengungguli baseline naif per-toko, dan apakah selisihnya signifikan?

In [ ]:
by_key = {(r["feature_set"], r["model"]): r for r in rows}
dm_rows = []
for variant, d in datasets.items():
    prop = by_key.get((variant, "LR-XGB (residual)"))
    naive = by_key.get((variant, "SeasonalNaive(store x dow x promo median)"))
    xgb_only = by_key.get((variant, "XGBoost"))
    for label, ref in [("SeasonalNaive", naive), ("XGBoost", xgb_only)]:
        if prop is None or ref is None:
            continue
        test = P.diebold_mariano(d.y_test, prop["_test_pred"], ref["_test_pred"])
        dm_rows.append({"varian": variant, "pembanding": label,
                        "RMSE usulan (log)": round(prop["test_RMSE"], 5),
                        "RMSE pembanding (log)": round(ref["test_RMSE"], 5),
                        "DM": round(test["DM"], 3),
                        "p_value": test["p_value"],
                        "usulan lebih baik": test["DM"] < 0})

dm_table = pd.DataFrame(dm_rows)
dm_table.to_csv(f"../results/{EXPERIMENT}_dm_test.csv", index=False)
dm_table

## 7. Pemeriksaan determinisme

In [ ]:
P.set_global_seed()
d = datasets["V1_customers_dropped"]
again = P.run_model("LR-XGB (residual)", P.fp_lr_xgb_residual, d, XGB_GRID,
                    inverse_transform=np.expm1)
first = by_key[("V1_customers_dropped", "LR-XGB (residual)")]
print("Prediksi identik bit-per-bit:",
      np.array_equal(again["_test_pred"], first["_test_pred"]))
print(f"RMSE (log): {again['test_RMSE']:.8f} vs {first['test_RMSE']:.8f}")
print("Hyperparameter terpilih identik:", again["params"] == first["params"])

## 8. Cara melaporkan hasil ini di naskah

* Baris `V0` **tidak boleh** muncul sebagai hasil metode usulan. Bila tetap
  ditampilkan, tandai secara eksplisit sebagai *leakage upper bound* dan jelaskan
  bahwa `Customers` tidak tersedia pada horizon peramalan.
* Angka Rossmann pada naskah versi sebelumnya (RMSE 577,63 / MSE 333.659,51 /
  RMSPE 0,06840 / R2 0,9651 / MAE 376,12) berasal dari konfigurasi `V0`. Angka
  tersebut harus diganti dengan hasil `V1`/`V2`, dan penurunan performanya dibahas
  terbuka - justru inilah kontribusi "empirical robustness" yang dijanjikan judul baru.
* Laporkan skala log dan skala asli pada dua blok tabel terpisah dengan judul kolom
  yang menyebut skalanya, dan sertakan RMSPE karena itu metrik resmi kompetisi.